# 04 — Visualizations

Clean visual storytelling for the final interactive dashboard. Focus: strongest insights, publication-ready aesthetics, dashboard-oriented figures.

In [6]:
# 1. Imports and helper

import pathlib
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

px.defaults.template = "plotly_white"

def apply_standard_layout(fig, height=500, margin_b=40):
    """Apply consistent dashboard-style layout to Plotly figures."""
    
    fig.update_layout(
        title_x=0.5,
        height=height,
        margin=dict(l=40, r=40, t=60, b=margin_b),
        font=dict(size=11),
        coloraxis_colorbar=dict(
            thickness=14,
            len=0.75
        )
    )
    
    return fig


# Load analytical dataset
project_root = pathlib.Path().resolve().parent

data_path = (
    project_root
    / 'data'
    / 'processed'
    / 'merged_analytical.csv'
)

df = pd.read_csv(data_path)

print(f"✓ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")

✓ Dataset loaded: 365 rows × 7 columns


## 2. Europe choropleth maps

Static and (optionally) animated maps showing `gap_pct` across Europe by year.

In [9]:
# Animated Europe choropleth

fig_anim = px.choropleth(
    df.dropna(subset=['gap_pct']),
    locations='iso3',
    color='gap_pct',
    animation_frame='year',

    scope='europe',
    projection='natural earth',

    color_continuous_scale=[
        [0.0, '#B22222'],   # red = negative gap
        [0.5, '#F5F5F5'],   # neutral
        [1.0, '#2E8B57']    # green = positive gap
    ],

    range_color=[-1, 1],

    title='Temporal evolution of EF proficiency gaps across Europe',

    hover_name='geo',

    hover_data={
        'year': True,
        'gap_pct': ':.2f',
        'learning_percentile': ':.2f',
        'ef_percentile': ':.2f'
    },

    labels={'gap_pct': 'Gap'}
)

fig_anim.update_geos(
    showcountries=True,
    countrycolor='white',
    showcoastlines=True,
    coastlinecolor='lightgray',
    showland=True,
    landcolor='#F8F8F8',
    fitbounds='locations'
)

fig_anim.update_layout(
    coloraxis_colorbar_title='Gap'
)

apply_standard_layout(fig_anim, height=620)

fig_anim.show()

## 3. Evolution over time

Animated choropleth showing year-by-year changes in the gap across Europe—reveals temporal patterns and geographic consistency.

In [11]:
# Italy vs European average through time

yearly_country = (
    df.groupby(['year', 'iso3'], as_index=False)['gap_pct']
    .mean()
)

europe_avg = (
    df.groupby('year', as_index=False)['gap_pct']
    .mean()
)

italy = yearly_country[
    yearly_country['iso3'] == 'ITA'
]

fig = go.Figure()

# Europe average
fig.add_trace(
    go.Scatter(
        x=europe_avg['year'],
        y=europe_avg['gap_pct'],
        mode='lines+markers',
        name='Europe average',
        line=dict(color='#2E8B57', width=3)
    )
)

# Italy
fig.add_trace(
    go.Scatter(
        x=italy['year'],
        y=italy['gap_pct'],
        mode='lines+markers',
        name='Italy',
        line=dict(color='#B22222', width=3)
    )
)

# Neutral reference
fig.add_hline(
    y=0,
    line_dash='dash',
    line_color='black',
    opacity=0.6
)

fig.update_layout(
    title='Italy compared with the European average EF proficiency gap',
    yaxis_title='Gap'
)

apply_standard_layout(fig, height=520)

fig.show()

## 4. Country performance comparison

Diverging bar chart showing all countries ranked by gap—green indicates outperformance relative to learning exposure, red underperformance.

In [12]:
# Strongest country-level contrasts

country_gap = (
    df.groupby(['iso3', 'geo'], as_index=False)['gap_pct']
    .mean()
    .dropna()
)

# Keep strongest positive and negative gaps
extremes = pd.concat([
    country_gap.nsmallest(8, 'gap_pct'),
    country_gap.nlargest(8, 'gap_pct')
])

# Sort for visual readability
extremes = extremes.sort_values('gap_pct')

fig_rank = px.bar(
    extremes,
    x='gap_pct',
    y='iso3',
    orientation='h',

    color='gap_pct',

    color_continuous_scale=[
        [0.0, '#B22222'],   # negative = red
        [0.5, '#F5F5F5'],   # neutral
        [1.0, '#2E8B57']    # positive = green
    ],

    range_color=[-1, 1],

    hover_name='geo',

    hover_data={
        'gap_pct': ':.2f',
        'iso3': False
    },

    title='Countries with the strongest EF proficiency gaps',

    labels={
        'gap_pct': 'Gap',
        'iso3': ''
    }
)

# Neutral reference line
fig_rank.add_vline(
    x=0,
    line_dash='dash',
    line_color='black',
    opacity=0.7
)

fig_rank.update_layout(
    coloraxis_colorbar_title='Gap'
)

apply_standard_layout(fig_rank, height=560)

fig_rank.show()

## 7. Dashboard reflection

**Visual storytelling for the interactive dashboard:**
- **Europe Choropleth** (Section 2): Central geographic anchor showing current average gap across all countries at a glance.
- **Animated Map** (Section 3): Reveals temporal evolution—are gaps widening, narrowing, or stable? Supports trend narrative.
- **Country Rankings** (Section 4): Unified diverging view—green shows strength, red shows gaps to address. Policy-ready visualization.
- **Policy Narrative** (Section 5): Italy vs Europe trajectory—demonstrates consistent underperformance and policy relevance. Easily adaptable to other countries.
- **Relationship Scatterplot** (Section 6): Nuanced exploratory view—points above diagonal exceed exposure expectations; below underperform. Hover details support investigation.

**Design principles implemented:**
- **Semantic color logic:** RdYlGn diverging scale where green=outperformance, red=underperformance. Consistent across all plots.
- **Interactive storytelling:** Animated map enables exploration; hover tooltips expose metadata.
- **Publication quality:** Clean layouts, clear titles, high contrast, professional aesthetic.
- **Dashboard-ready:** Each figure is self-contained, exportable, and suitable for interactive filtering/parameter changes.
- **Policy-oriented:** Emphasis on country performance narrative, gap interpretation, and actionable insights for stakeholders.